In [1]:
# === SESSION BOOTSTRAP (run first in every notebook) ======================
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys

PARENT = "/content/drive/MyDrive/UAV_TRUST_Research"
REPO   = f"{PARENT}/uav-trust-research"

for fname in (".gitconfig", ".git-credentials"):
    src = os.path.join(PARENT, fname)
    if os.path.exists(src):
        subprocess.run(f'cp "{src}" /root/{fname}', shell=True)
subprocess.run("git config --global credential.helper store", shell=True)

r = subprocess.run("git config --global user.name && git config --global user.email",
                   shell=True, capture_output=True, text=True)
print("git identity:", r.stdout.strip() or "MISSING - run 00_setup.ipynb first")

if os.path.isdir(REPO):
    os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    print("cwd:", os.getcwd())
else:
    print("Repository not on Drive yet - run 00_setup.ipynb first.")

Mounted at /content/drive
git identity: Md Anas Biswas
anasbiswas@gmail.com
cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research


In [2]:
!pip install xgboost scikit-learn matplotlib pandas numpy scipy requests --quiet

In [3]:
# Configuration for UAV-CAS (second dataset; same pipeline as notebook 03)
CONFIG = {
    "data_dir": "data/uav_cas",
    "label_col": None,               # auto-detect; override after seeing the schema printout
    "normal_value": None,            # auto-detect; override after seeing the schema printout
    "include_families": None,        # set to the single attack families after inspecting the schema
    "drop_col_patterns": ["unnamed", "flowid", "flow_id", "srcaddr", "dstaddr",
                           "src_ip", "dst_ip", "srcport", "dstport", "mac", "index", "timestamp"],
    "seeds": list(range(10)),
    "conformal_alpha": 0.10,
    "n_ece_bins": 15,
    "normal_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.10, "test_shift": 0.10},
    "family_fracs": {"train": 0.60, "cal": 0.20, "test_seen": 0.20},
    "xgb": {"n_estimators": 300, "max_depth": 6, "learning_rate": 0.1,
            "subsample": 0.9, "colsample_bytree": 0.9, "tree_method": "hist"},
    "fig_dir": "figures",
    "report_dir": "reports",
}
for d in [CONFIG["data_dir"], CONFIG["fig_dir"], CONFIG["report_dir"]]:
    os.makedirs(d, exist_ok=True)
CONFIG

{'data_dir': 'data/uav_cas',
 'label_col': None,
 'normal_value': None,
 'include_families': None,
 'drop_col_patterns': ['unnamed',
  'flowid',
  'flow_id',
  'srcaddr',
  'dstaddr',
  'src_ip',
  'dst_ip',
  'srcport',
  'dstport',
  'mac',
  'index',
  'timestamp'],
 'seeds': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9],
 'conformal_alpha': 0.1,
 'n_ece_bins': 15,
 'normal_fracs': {'train': 0.6,
  'cal': 0.2,
  'test_seen': 0.1,
  'test_shift': 0.1},
 'family_fracs': {'train': 0.6, 'cal': 0.2, 'test_seen': 0.2},
 'xgb': {'n_estimators': 300,
  'max_depth': 6,
  'learning_rate': 0.1,
  'subsample': 0.9,
  'colsample_bytree': 0.9,
  'tree_method': 'hist'},
 'fig_dir': 'figures',
 'report_dir': 'reports'}

In [4]:
# Imports (shared logic in src/)
import numpy as np, pandas as pd, glob
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.metrics import (accuracy_score, f1_score,
                             balanced_accuracy_score, recall_score)
from src.trust import (top_label_ece, brier_binary, conformal_qhat, coverage_size,
                       aurc, logit, fit_calibrators, apply_calibrators)
from src.data import load_csvs, detect_schema, prepare_splits
print("imports ready")

imports ready


In [8]:
import os
print(os.listdir("data/uav_cas") if os.path.isdir("data/uav_cas") else "still missing")

[]


In [7]:
import glob, os
print("cwd:", os.getcwd())
print("target dir exists:", os.path.isdir("data/uav_cas"))
print("contents of data/uav_cas:", os.listdir("data/uav_cas") if os.path.isdir("data/uav_cas") else "MISSING")
# widen the search: any UAV-CAS-like file anywhere under the repo and common Drive spots
for root in ["data", "/content/drive/MyDrive", "/content"]:
    hits = glob.glob(f"{root}/**/*uav*cas*", recursive=True) + \
           glob.glob(f"{root}/**/*UAV*CAS*", recursive=True)
    if hits: print(root, "->", hits[:10])

cwd: /content/drive/MyDrive/UAV_TRUST_Research/uav-trust-research
target dir exists: True
contents of data/uav_cas: []
data -> ['data/uav_cas']


KeyboardInterrupt: 

In [6]:
# UAV-CAS is on IEEE DataPort and requires a free IEEE sign-in to download.
# Download the released CSV(s) from:
#   https://ieee-dataport.org/documents/uav-cas-calibrated-digital-twin-dataset-intrusion-detection-uav-swarm-networks
# and place them in CONFIG["data_dir"] (data/uav_cas/), then run this cell.
csvs = glob.glob(os.path.join(CONFIG["data_dir"], "**/*.csv"), recursive=True)
assert csvs, "Place the UAV-CAS CSV(s) in %s and re-run this cell." % CONFIG["data_dir"]
df = load_csvs(CONFIG["data_dir"])
label_col, normal_value, families = detect_schema(df, CONFIG["label_col"], CONFIG["normal_value"])
print("shape:", df.shape)
print("label column:", label_col)
print("normal value:", repr(normal_value))
print("class distribution:")
for k, v in df[label_col].value_counts().items():
    print("   %-28s %d" % (k, v))

AssertionError: Place the UAV-CAS CSV(s) in data/uav_cas and re-run this cell.

In [ ]:
# Restrict to the attack families you want to hold out (e.g. the single-attack families,
# excluding collaborative compositions). Set CONFIG["include_families"] from the schema above.
if CONFIG["include_families"]:
    keep = [normal_value] + list(CONFIG["include_families"])
    df = df[df[label_col].isin(keep)].reset_index(drop=True)
    families = list(CONFIG["include_families"])
print("families to hold out in turn:", families)

In [ ]:
# Single held-out-family evaluation for one seed (returns a metrics dict)
def run_once(df, label_col, normal_value, F, seed, cfg):
    S = prepare_splits(df, label_col, normal_value, F, cfg["drop_col_patterns"],
                       cfg["normal_fracs"], cfg["family_fracs"], seed)
    clf = xgb.XGBClassifier(objective="binary:logistic", eval_metric="logloss",
                            random_state=seed, **cfg["xgb"])
    clf.fit(S["X_train"], S["y_train"])

    def sc(X):
        p = clf.predict_proba(X)[:, 1]; return p, logit(p)
    p_cal, lg_cal   = sc(S["X_cal"])
    p_seen, lg_seen = sc(S["X_seen"])
    p_shift, lg_shift = sc(S["X_shift"])
    y_seen, y_shift = S["y_seen"], S["y_shift"]

    fitted = fit_calibrators(lg_cal, p_cal, S["y_cal"])
    cs = apply_calibrators(fitted, lg_seen, p_seen)
    ch = apply_calibrators(fitted, lg_shift, p_shift)
    qhat = conformal_qhat(p_cal, S["y_cal"], alpha=cfg["conformal_alpha"])
    pr_s = (p_seen >= 0.5).astype(int)
    pr_h = (p_shift >= 0.5).astype(int)
    nb = cfg["n_ece_bins"]
    return {
        "held_out": F, "seed": seed,
        "shift_heldout_recall": recall_score(y_shift, pr_h, pos_label=1),
        "shift_trivial_base": max(np.mean(y_shift == 0), np.mean(y_shift == 1)),
        "seen_balacc": balanced_accuracy_score(y_seen, pr_s),
        "shift_balacc": balanced_accuracy_score(y_shift, pr_h),
        "seen_macroF1": f1_score(y_seen, pr_s, average="macro"),
        "shift_macroF1": f1_score(y_shift, pr_h, average="macro"),
        "seen_ECE_temp": top_label_ece(cs["temperature"], y_seen, nb),
        "shift_ECE_temp": top_label_ece(ch["temperature"], y_shift, nb),
        "shift_Brier_temp": brier_binary(ch["temperature"], y_shift),
        "seen_coverage": coverage_size(p_seen, y_seen, qhat)[0],
        "shift_coverage": coverage_size(p_shift, y_shift, qhat)[0],
        "seen_AURC": aurc(np.maximum(p_seen, 1 - p_seen), (pr_s == y_seen).astype(float))[0],
        "shift_AURC": aurc(np.maximum(p_shift, 1 - p_shift), (pr_h == y_shift).astype(float))[0],
        "T": fitted["temperature"],
    }

In [ ]:
# Run every family across every seed
rows = []
for seed in CONFIG["seeds"]:
    for F in families:
        rows.append(run_once(df, label_col, normal_value, F, seed, CONFIG))
    print("seed done:", seed)
raw = pd.DataFrame(rows)
print("runs:", raw.shape[0], "= %d seeds x %d families" % (len(CONFIG["seeds"]), len(families)))

In [ ]:
# Aggregate across seeds: mean and std per family
key = ["shift_balacc", "shift_macroF1", "shift_heldout_recall",
       "shift_ECE_temp", "shift_coverage", "shift_AURC", "seen_coverage", "T"]
agg = raw.groupby("held_out")[key].agg(["mean", "std"])
summary = pd.DataFrame(index=agg.index)
for k in key:
    summary[k] = (agg[k]["mean"].round(4).astype(str) + " ± " + agg[k]["std"].round(4).astype(str))
print(summary.to_string())
raw.to_csv(os.path.join(CONFIG["report_dir"], "04_uavcas_seed_raw.csv"), index=False)
flat = agg.copy(); flat.columns = ["%s_%s" % (a, b) for a, b in flat.columns]
flat.round(6).to_csv(os.path.join(CONFIG["report_dir"], "04_uavcas_seed_aggregate.csv"))
print("saved 04_uavcas_seed_raw.csv and 04_uavcas_seed_aggregate.csv")

In [ ]:
# Figure: per-family spread across seeds on UAV-CAS (does the UAVIDS pattern replicate?)
fams = list(agg.index)
labels = [str(f).replace(" Attack", "").replace("_", " ") for f in fams]
panels = [("shift_balacc", "Balanced accuracy (shift)", None),
          ("shift_ECE_temp", "ECE temperature (shift)", None),
          ("shift_coverage", "Conformal coverage (shift)", 1 - CONFIG["conformal_alpha"])]
jit = np.random.default_rng(0)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
for a, (col, title, target) in zip(ax, panels):
    for i, F in enumerate(fams):
        v = raw.loc[raw["held_out"] == F, col].values
        a.scatter(i + jit.uniform(-0.12, 0.12, len(v)), v, s=20, alpha=0.6, color="#1565c0")
        a.scatter([i], [v.mean()], marker="_", s=700, color="black", zorder=3)
    a.set_xticks(range(len(fams))); a.set_xticklabels(labels, rotation=20, ha="right")
    a.set_title(title)
    if target is not None:
        a.axhline(target, ls="--", color="gray", lw=1)
fig.suptitle("UAV-CAS  |  each family held out across %d seeds  |  black bar = mean" % len(CONFIG["seeds"]))
fig.tight_layout()
fig.savefig(os.path.join(CONFIG["fig_dir"], "04_uavcas_seed_spread.png"), dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# Commit results (end-of-unit discipline)
!git add reports/ figures/ notebooks/
!git commit -m "04 UAV-CAS: external-validity replication of multi-family held-out trust evaluation across seeds"
!git push origin main